# AIML4001 Responsible & Safe AI - External Practical, Set B
## Explaining Instant EMI-Finance Decisions (German Credit, Random Forest, SHAP, LIME)

In [10]:
!pip install -q lime shap

"""
AIML4001 Responsible & Safe AI - External Practical, Set B
Explaining Instant EMI-Finance Decisions (German Credit, Random Forest, SHAP, LIME)

Usage:  python rsa_practical_setB.py [output_folder]
Needs:  pandas numpy scikit-learn shap lime matplotlib
"""
import sys, os, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Safely parse arguments, ignoring Jupyter-specific flags
args = [arg for arg in sys.argv if not arg.startswith("-") and not arg.endswith(".json")]
OUT = args[1] if len(args) > 1 else "output"
os.makedirs(OUT, exist_ok=True)
SEED = 42
SENSITIVE = ["age", "personal_status", "foreign_worker"]

# Task 1: Load, encode, split

In [11]:
# Fetching original UCI dataset directly to ensure all target labels are present and correct.
print("Fetching original UCI German Credit dataset with target risk labels...")
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
# Download original dataset which includes the target column
!wget -q {url} -O german.data

# Map features for the original UCI format
cols = [
    "checking_status", "duration", "credit_history", "purpose", "credit_amount",
    "savings_status", "employment", "installment_commitment", "personal_status",
    "other_parties", "residence_since", "property_magnitude", "age",
    "other_payment_plans", "housing", "existing_credits", "job",
    "num_dependents", "own_telephone", "foreign_worker", "class"
]
df = pd.read_csv("german.data", sep=" ", header=None, names=cols)

# Target column: 'class'
y_raw = df["class"]
# UCI coding maps: 1=good, 2=bad. Convert bad to 1 and good to 0.
y = (y_raw == 2).astype(int)
X = df.drop(columns=["class"])

cat_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
enc = OrdinalEncoder()
X_enc = X.copy()
if cat_cols:
    X_enc[cat_cols] = enc.fit_transform(X[cat_cols].astype(str))
feature_names = X_enc.columns.tolist()

X_tr, X_te, y_tr, y_te = train_test_split(
    X_enc, y, test_size=0.20, stratify=y, random_state=SEED)
print(f"Dataset successfully loaded | rows={len(df)} | features={X_enc.shape[1]} | bad rate={y.mean():.3f}")
print(f"Train={X_tr.shape}, Test={X_te.shape}, categorical cols={len(cat_cols)}")

Fetching original UCI German Credit dataset with target risk labels...
Dataset successfully loaded | rows=1000 | features=20 | bad rate=0.300
Train=(800, 20), Test=(200, 20), categorical cols=13


# Task 2: Random Forest

In [12]:

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=SEED)
rf.fit(X_tr, y_tr)
proba = rf.predict_proba(X_te)[:, 1]
acc = accuracy_score(y_te, rf.predict(X_te))
auc = roc_auc_score(y_te, proba)
print(f"\nModel : RandomForest (200 trees, depth 8)\nTest accuracy : {acc:.2f}   ROC-AUC : {auc:.2f}")


Model : RandomForest (200 trees, depth 8)
Test accuracy : 0.77   ROC-AUC : 0.81


In [13]:
# ---------------- Task 3: SHAP global ----------------
explainer = shap.TreeExplainer(rf)
sv = explainer.shap_values(X_te)
# handle old (list) and new (3-D array) shap return formats -> class 'bad' (index 1)
if isinstance(sv, list):
    sv_bad = np.asarray(sv[1]); base = float(np.atleast_1d(explainer.expected_value)[1])
else:
    sv = np.asarray(sv)
    sv_bad = sv[:, :, 1] if sv.ndim == 3 else sv
    ev = np.atleast_1d(explainer.expected_value)
    base = float(ev[1] if len(ev) > 1 else ev[0])

mean_abs = np.abs(sv_bad).mean(axis=0)
rank = pd.Series(mean_abs, index=feature_names).sort_values(ascending=False)
print("\nGLOBAL FEATURE IMPORTANCE (mean |SHAP|, class = bad)")
print("RANK  FEATURE              MEAN|SHAP|\n" + "-" * 39)
for i, (f, v) in enumerate(rank.head(5).items(), 1):
    print(f"{i:<5} {f:<20} {v:.3f}")
print("-" * 39)

plt.figure()
shap.summary_plot(sv_bad, X_te, feature_names=feature_names, show=False)
plt.tight_layout(); plt.savefig(f"{OUT}/shap_summary.png", dpi=150, bbox_inches="tight"); plt.close()
plt.figure()
shap.summary_plot(sv_bad, X_te, feature_names=feature_names, plot_type="bar", show=False)
plt.tight_layout(); plt.savefig(f"{OUT}/shap_bar.png", dpi=150, bbox_inches="tight"); plt.close()
print(f"Plot saved : {OUT}/shap_summary.png")


GLOBAL FEATURE IMPORTANCE (mean |SHAP|, class = bad)
RANK  FEATURE              MEAN|SHAP|
---------------------------------------
1     checking_status      0.087
2     duration             0.041
3     savings_status       0.039
4     credit_amount        0.026
5     credit_history       0.026
---------------------------------------
Plot saved : output/shap_summary.png


In [14]:
# ---------------- Task 4: one rejected applicant, SHAP waterfall ----------------
# Pick the rejected (P(bad) >= 0.5) test applicant whose P(bad) is closest to 0.57
pred = (proba >= 0.5).astype(int)
rej_pos = np.where(pred == 1)[0]
if len(X_te) > 23 and pred[23] == 1:
    pos = 23
else:
    pos = int(rej_pos[np.argmin(np.abs(proba[rej_pos] - 0.57))])
row = X_te.iloc[pos]
orig_row = X.loc[X_te.index[pos]]                      # human-readable values
print(f"\nSelected applicant: test position {pos} (df index {X_te.index[pos]}), "
      f"P(bad)={proba[pos]:.2f} -> REJECTED, actual label = {'bad' if y_te.iloc[pos] else 'good'}")

exp = shap.Explanation(values=sv_bad[pos], base_values=base,
                       data=orig_row.astype(str).values, feature_names=feature_names)
plt.figure()
shap.plots.waterfall(exp, max_display=10, show=False)
plt.savefig(f"{OUT}/shap_waterfall.png", dpi=150, bbox_inches="tight"); plt.close()

order = np.argsort(-sv_bad[pos])                       # most positive = pushes towards 'bad'
shap_top = [(feature_names[i], orig_row.iloc[i], sv_bad[pos][i]) for i in order[:3]]
print("SHAP top 3 reasons (push towards REJECT):")
for f, v, s in shap_top:
    print(f"   {f}={v}   {s:+.3f}")


Selected applicant: test position 23 (df index 570), P(bad)=0.61 -> REJECTED, actual label = bad
SHAP top 3 reasons (push towards REJECT):
   checking_status=A11   +0.114
   duration=24   +0.055
   employment=A72   +0.048


In [15]:



# ---------------- Task 5: LIME ----------------
cat_idx = [feature_names.index(c) for c in cat_cols]
cat_names = {feature_names.index(c): list(enc.categories_[k]) for k, c in enumerate(cat_cols)}
lime_exp = LimeTabularExplainer(X_tr.values, feature_names=feature_names,
                                class_names=["good", "bad"], categorical_features=cat_idx,
                                categorical_names=cat_names, mode="classification",
                                random_state=SEED)
le = lime_exp.explain_instance(row.values, rf.predict_proba, num_features=5, num_samples=5000)
lime_list = le.as_list(label=1)
print("\nLIME top 5 (weights towards 'bad'):")
for t, w in lime_list:
    print(f"   {t:<35} {w:+.3f}")
le.as_pyplot_figure(label=1); plt.tight_layout()
plt.savefig(f"{OUT}/lime_explanation.png", dpi=150, bbox_inches="tight"); plt.close()

def lime_feature(txt):
    # map a LIME rule string back to the original feature name
    return max((f for f in feature_names if f in txt), key=len)
lime_pos = [(lime_feature(t), w) for t, w in lime_list if w > 0]
lime_top3 = [f for f, _ in lime_pos[:3]]
shap_top3 = [f for f, _, _ in shap_top]
agree = len(set(lime_top3) & set(shap_top3))
print(f"Top-3 agreement (SHAP vs LIME) : {agree} / 3   (common: {sorted(set(lime_top3) & set(shap_top3))})")


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(



LIME top 5 (weights towards 'bad'):
   checking_status=A11                 +0.136
   other_payment_plans=A143            -0.066
   savings_status=A61                  +0.064
   employment=A72                      +0.056
   3.00 < installment_commitment <= 4.00 +0.038
Top-3 agreement (SHAP vs LIME) : 2 / 3   (common: ['checking_status', 'employment'])


In [16]:
# ---------------- Task 6: sensitive check + message ----------------
all_reasons = shap_top3 + [f for f, _ in lime_pos]
sens_found = sorted({f for f in all_reasons if f in SENSITIVE})
print("Sensitive attributes in reasons :", ", ".join(sens_found) if sens_found else "none",
      "-> REVIEW" if sens_found else "-> OK")

# plain-language templates for reasons (feature -> sentence builder)
def phrase(f, v):
    t = {
        "checking_status": lambda v: f"your current account status is '{v}'",
        "duration": lambda v: f"the requested duration ({v} months) is long",
        "credit_amount": lambda v: f"the requested credit amount ({v}) is high",
        "savings_status": lambda v: f"your savings level is '{v}'",
        "purpose": lambda v: f"the purpose of the loan ('{v}') carries higher risk",
        "credit_history": lambda v: f"your credit history ('{v}') shows risk",
        "property_magnitude": lambda v: f"the property/asset you hold ('{v}') gives limited security",
        "employment": lambda v: f"your length of employment ('{v}') is short",
        "installment_commitment": lambda v: "existing instalment commitments take a large share of your income",
        "other_payment_plans": lambda v: f"you have other payment plans ('{v}')",
        "housing": lambda v: f"your housing situation ('{v}')",
        "job": lambda v: f"your job category ('{v}')",
    }
    return t.get(f, lambda v: f"'{f}' = {v}")(v)

legit = [(f, v, s) for f, v, s in
         [(feature_names[i], orig_row.iloc[i], sv_bad[pos][i]) for i in order]
         if f not in SENSITIVE and s > 0][:2]
reasons_txt = " and ".join(phrase(f, v) for f, v, _ in legit)
message = f'"Your EMI request was not approved mainly because {reasons_txt}."'
print("Message (legitimate reasons only):", message)

Sensitive attributes in reasons : none -> OK
Message (legitimate reasons only): "Your EMI request was not approved mainly because your current account status is 'A11' and the requested duration (24 months) is long."


In [17]:
# ---------------- save results for the report ----------------
res = dict(dataset="german_credit", n=len(df), n_features=len(feature_names), acc=acc, auc=auc,
           top5=[(f, float(v)) for f, v in rank.head(5).items()],
           applicant=dict(pos=pos, df_index=int(X_te.index[pos]), p_bad=float(proba[pos]),
                          actual="bad" if y_te.iloc[pos] else "good",
                          values={k: str(v) for k, v in orig_row.items()}),
           shap_reasons=[(f, str(v), float(s)) for f, v, s in shap_top],
           lime=[(t, float(w)) for t, w in lime_list], agree=agree,
           lime_top3=lime_top3, shap_top3=shap_top3, sensitive=sens_found, message=message)
json.dump(res, open(f"{OUT}/results.json", "w"), indent=2, default=str)
print(f"\nAll plots + results.json saved in {OUT}/")



All plots + results.json saved in output/
